##  Carga de las fuentes de información

In [2]:
# Instala las librerías a usar
!pip install -q gdown
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [3]:
import gdown
import zipfile
import os
import pandas as pd
import json
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import CharacterTextSplitter

###  Descarga del dataset desde Google Drive

In [4]:
file_id = "1LY9FWZzuB-KSrdWkHAMDtNk2LemWLxls" #ID del archivo zip en google
output = "fuentes_de_informacion.zip"

gdown.download(id=file_id, output=output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1LY9FWZzuB-KSrdWkHAMDtNk2LemWLxls
To: /content/fuentes_de_informacion.zip
100%|██████████| 3.81M/3.81M [00:00<00:00, 186MB/s]


'fuentes_de_informacion.zip'

### Descompresión del archivo ZIP

In [5]:
# Carpeta donde vamos a descomprimir todo
extract_path = "/content/fuentes_de_informacion"

# Crea la carpeta si no existe
os.makedirs(extract_path, exist_ok=True)

# Abro el ZIP y extraigo todo adentro de extract_path
with zipfile.ZipFile(output, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP descomprimido")

ZIP descomprimido


In [6]:
# Recorremos las carpetas para ver qué se descomprimió
for root, dirs, files in os.walk(extract_path):

    # Esto es solo para que se imprima ordenado visualmente
    nivel = root.replace(extract_path, "").count(os.sep)
    indent = "  " * nivel

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (nivel + 1)

    # Mostramos solo algunos archivos para no llenar todo con miles de reseñas
    for file in files[:5]:
        print(f"{subindent}{file}")

    if len(files) > 5:
        print(f"{subindent}... ({len(files)} archivos)")

fuentes_de_informacion/
  fuentes_de_informacion/
    productos.xlsx
    vendedores.csv
    inventario_sucursales.csv
    tickets_soporte.csv
    devoluciones.csv
    ... (8 archivos)
    manuales_productos/
      manual_P0149_Aire_Split.md
      manual_P0194_Max_Ventilador_de_Torre.md
      manual_P0017_Profesional_Batidora_de_Mano.md
      manual_P0058_Eco_Tostadora.md
      manual_P0296_Planchita_de_Pelo.md
      ... (50 archivos)
    resenas_usuarios/
      resena_R01437.txt
      resena_R01300.txt
      resena_R01235.txt
      resena_R02219.txt
      resena_R04681.txt
      ... (5015 archivos)


In [7]:
# Ruta base real del dataset.
# Quedó una carpeta fuentes_de_informacion adentro de otra por cómo estaba armado el ZIP.
base_path = Path("/content/fuentes_de_informacion/fuentes_de_informacion")

# Rutas a las carpetas de textos
resenas_path = base_path / "resenas_usuarios"
manuales_path = base_path / "manuales_productos"

# Rutas a archivos principales
productos_csv_path = base_path / "productos.csv"
productos_xlsx_path = base_path / "productos.xlsx"
inventario_path = base_path / "inventario_sucursales.csv"
ventas_path = base_path / "ventas_historicas.csv"
devoluciones_path = base_path / "devoluciones.csv"
tickets_path = base_path / "tickets_soporte.csv"
vendedores_path = base_path / "vendedores.csv"
faqs_path = base_path / "faqs.json"

# Verificamos rápido que las rutas existan
print("Base:", base_path.exists())
print("Reseñas:", resenas_path.exists())
print("Manuales:", manuales_path.exists())
print("Productos CSV:", productos_csv_path.exists())
print("FAQs:", faqs_path.exists())

Base: True
Reseñas: True
Manuales: True
Productos CSV: True
FAQs: True


In [8]:
# Cargamos las tablas principales
productos_df = pd.read_csv(productos_csv_path)
inventario_df = pd.read_csv(inventario_path)
ventas_df = pd.read_csv(ventas_path)
devoluciones_df = pd.read_csv(devoluciones_path)
tickets_df = pd.read_csv(tickets_path)
vendedores_df = pd.read_csv(vendedores_path)

# Vemos cuántas filas y columnas tiene cada tabla
print("productos:", productos_df.shape)
print("inventario:", inventario_df.shape)
print("ventas:", ventas_df.shape)
print("devoluciones:", devoluciones_df.shape)
print("tickets:", tickets_df.shape)
print("vendedores:", vendedores_df.shape)

productos: (300, 14)
inventario: (4100, 14)
ventas: (10000, 15)
devoluciones: (800, 14)
tickets: (2000, 17)
vendedores: (100, 10)


##  Diseño de las fuentes de datos

El sistema utilizará tres fuentes de conocimiento con propósitos distintos:

- Base vectorial: para recuperar información textual no estructurada mediante similitud semántica.
- Base tabular: para responder consultas que requieren filtros, comparaciones, rangos o agregaciones.
- Base de grafos: para representar relaciones entre productos, categorías, subcategorías y marcas.

Esta separación va a permitir elegir la fuente más adecuada según la intención de la consulta del usuario.

In [10]:
# Resumen de diseño de las fuentes que vamos a usar en el sistema

disenio_fuentes = pd.DataFrame([
    {
        "fuente": "Base vectorial",
        "motor": "ChromaDB",
        "informacion": "Manuales, FAQs, reseñas y descripciones textuales de tickets de soporte",
        "uso": "Preguntas sobre uso de productos, opiniones, problemas frecuentes y respuestas en lenguaje natural"
    },
    {
        "fuente": "Base tabular",
        "motor": "Pandas",
        "informacion": "Productos, inventario, ventas, devoluciones, tickets y vendedores",
        "uso": "Consultas con filtros, precios, stock, categorías, ventas, devoluciones y métricas"
    },
    {
        "fuente": "Base de grafos",
        "motor": "GrafitoDB",
        "informacion": "Relaciones entre productos, categorías, subcategorías y marcas",
        "uso": "Consultas sobre relaciones, productos conectados y navegación por categorías"
    }
])

display(disenio_fuentes)

,fuente,motor,informacion,uso
0,Base vectorial,ChromaDB,"Manuales, FAQs, reseñas y descripciones textua...","Preguntas sobre uso de productos, opiniones, p..."
1,Base tabular,Pandas,"Productos, inventario, ventas, devoluciones, t...","Consultas con filtros, precios, stock, categor..."
2,Base de grafos,GrafitoDB,"Relaciones entre productos, categorías, subcat...","Consultas sobre relaciones, productos conectad..."


##  Preparación de documentos para la base vectorial

In [11]:
documentos_vectoriales = []

# 1) FAQs
# Las FAQs sirven para preguntas frecuentes directas de usuarios.
with open(faqs_path, "r", encoding="utf-8") as f:
    faqs_data = json.load(f)

for i, faq in enumerate(faqs_data):
    texto = " ".join([str(v) for v in faq.values()])

    documentos_vectoriales.append({
        "id": f"faq_{i}",
        "texto": texto,
        "metadata": {
            "fuente": "faqs",
            "tipo": "pregunta_frecuente"
        }
    })

# 2) Manuales de productos
# Los manuales sirven para consultas sobre uso, mantenimiento y especificaciones.
for archivo in manuales_path.glob("*.md"):
    with open(archivo, "r", encoding="utf-8") as f:
        texto = f.read()

    documentos_vectoriales.append({
        "id": f"manual_{archivo.stem}",
        "texto": texto,
        "metadata": {
            "fuente": "manuales_productos",
            "tipo": "manual",
            "archivo": archivo.name
        }
    })

# 3) Reseñas de usuarios
# Las reseñas sirven para preguntas sobre opiniones y experiencia de usuarios.
for archivo in resenas_path.glob("*.txt"):
    with open(archivo, "r", encoding="utf-8") as f:
        texto = f.read()

    documentos_vectoriales.append({
        "id": f"resena_{archivo.stem}",
        "texto": texto,
        "metadata": {
            "fuente": "resenas_usuarios",
            "tipo": "resena",
            "archivo": archivo.name
        }
    })

# 4) Tickets de soporte
# Usamos la descripción textual de los tickets para recuperar problemas similares.
for _, row in tickets_df.iterrows():
    texto = f"""
    Producto: {row['nombre_producto']}
    Tipo de problema: {row['tipo_problema']}
    Descripción: {row['descripcion']}
    Severidad: {row['severidad']}
    Categoría: {row['categoria']}
    Estado: {row['estado']}
    Garantía válida: {row['garantia_valida']}
    """

    documentos_vectoriales.append({
        "id": f"ticket_{row['id_ticket']}",
        "texto": texto,
        "metadata": {
            "fuente": "tickets_soporte",
            "tipo": "ticket",
            "id_producto": row["id_producto"],
            "nombre_producto": row["nombre_producto"],
            "categoria": row["categoria"]
        }
    })
print("Documentos preparados para ChromaDB:", len(documentos_vectoriales))


Documentos preparados para ChromaDB: 10065


## Base de datos vectorial con ChromaDB

In [12]:
# Modelo multilingüe apto para español.
# Es chico, rápido y sirve para búsqueda semántica.
embedding_model_name = "intfloat/multilingual-e5-small"

embedding_model = SentenceTransformer(embedding_model_name)

print("Modelo cargado:", embedding_model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Modelo cargado: intfloat/multilingual-e5-small


###  Segmentación de documentos

In [13]:
# Dividimos los textos largos en fragmentos más chicos.
# Esto ayuda a que ChromaDB recupere partes más puntuales y no documentos enormes.

text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=700,
    chunk_overlap=100
)

fragmentos_vectoriales = []

for doc in documentos_vectoriales:
    partes = text_splitter.split_text(doc["texto"])

    for i, parte in enumerate(partes):
        fragmentos_vectoriales.append({
            "id": f"{doc['id']}_chunk_{i}",
            "texto": parte,
            "metadata": {
                **doc["metadata"],
                "doc_id": doc["id"],
                "chunk": i
            }
        })

print("Documentos originales:", len(documentos_vectoriales))
print("Fragmentos generados:", len(fragmentos_vectoriales))

Documentos originales: 10065
Fragmentos generados: 10575


### Creación de la colección en ChromaDB

In [14]:
# Creamos el cliente local de ChromaDB.
# El cliente es el objeto que administra las colecciones.
client = chromadb.Client()

# Nombre de la colección donde vamos a guardar los fragmentos.
collection_name = "electrodomesticos_docs"

# Si la colección ya existía de una ejecución anterior, la borra.
# Esto evita cargar documentos duplicados si volvemos a correr la celda.
try:
    client.delete_collection(name=collection_name)
except:
    pass

# Creamos la colección nueva.
collection = client.create_collection(name=collection_name)

print("Colección creada:", collection_name)

Colección creada: electrodomesticos_docs


### Generación de embeddings

In [15]:
# Separamos la información en listas porque ChromaDB trabaja con listas paralelas:
# ids: identificadores únicos
# documents: textos
# metadatas: información extra de cada fragmento

ids = [frag["id"] for frag in fragmentos_vectoriales]
documents = [frag["texto"] for frag in fragmentos_vectoriales]
metadatas = [frag["metadata"] for frag in fragmentos_vectoriales]

# Como usamos el modelo E5, agregamos "passage:" delante de los documentos.
# Esto ayuda al modelo a entender que estos textos son pasajes a recuperar.
documents_for_embedding = ["passage: " + texto for texto in documents]

# Generamos los embeddings.
# batch_size indica cuántos textos procesa juntos.
# normalize_embeddings=True deja los vectores normalizados para comparación semántica.
embeddings = embedding_model.encode(
    documents_for_embedding,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
).tolist()

print("Embeddings generados:", len(embeddings))

Batches:   0%|          | 0/166 [00:00<?, ?it/s]

KeyboardInterrupt: 